# 03 — Markov Blanket Intuition

**Companion to `docs/03_graphical_models.md` & `docs/04_markov_blanket_theory.md`
(Phases 2–3).**

The Markov Blanket of a target `T` is the minimal set making `T` independent of
everything else:

$$ T \perp (V \setminus (\{T\}\cup MB(T))) \mid MB(T) $$

In a faithful Bayesian network:

$$ MB(T) = \text{parents}(T) \cup \text{children}(T) \cup \text{spouses}(T) $$

The **spouse** term is the subtle one — this notebook makes it concrete.


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
rng = np.random.default_rng(1)
ALPHA = 0.05

## 0. Helpers (inline G-test + MI)

Same self-contained tools as notebooks 01–02, repeated so this notebook runs on
its own.


In [3]:
def _counts(*cols):
    data = np.column_stack([np.asarray(c).reshape(-1) for c in cols])
    _, inv = np.unique(data, axis=0, return_inverse=True)
    return np.bincount(inv)

def entropy(*cols):
    p = _counts(*cols); p = p / p.sum()
    return float(-np.sum(p * np.log(p + (p == 0))))

def mutual_information(x, y):
    return max(0.0, entropy(x) + entropy(y) - entropy(x, y))

def _table(x, y):
    xs, xi = np.unique(x, return_inverse=True)
    ys, yi = np.unique(y, return_inverse=True)
    t = np.zeros((len(xs), len(ys))); np.add.at(t, (xi, yi), 1); return t

def g_test(x, y, z=None):
    z = z or []; x = np.asarray(x); y = np.asarray(y)
    if not z:
        tabs = [_table(x, y)]
    else:
        zmat = np.column_stack([np.asarray(c).reshape(-1) for c in z])
        _, inv = np.unique(zmat, axis=0, return_inverse=True)
        tabs = [_table(x[inv==s], y[inv==s]) for s in range(inv.max()+1)]
    G, dof = 0.0, 0
    for tab in tabs:
        n = tab.sum()
        if n == 0: continue
        row = tab.sum(1, keepdims=True); col = tab.sum(0, keepdims=True)
        e = row @ col / n; nz = (tab > 0) & (e > 0)
        G += 2.0 * np.sum(tab[nz] * np.log(tab[nz] / e[nz]))
        dof += max(int(np.count_nonzero(row))-1,0) * max(int(np.count_nonzero(col))-1,0)
    dof = max(dof, 1)
    return float(G), dof, float(chi2.sf(G, dof))

## 1. Build a network where T has a spouse

```
    A → T → C ← S
            ↑
            B → C
```

- **Parent** of T: `A`   ·   **Child** of T: `C`
- **Spouses** of T: `B`, `S` (the *other* parents of T's child C)
- `D`: independent noise, NOT in the blanket

True Markov Blanket: `{A, C, B, S}`.


In [4]:
n = 10000
A = rng.integers(0, 2, n)
B = rng.integers(0, 2, n)
S = rng.integers(0, 2, n)
D = rng.integers(0, 2, n)                       # irrelevant
T = (rng.random(n) < np.where(A == 1, 0.8, 0.2)).astype(int)      # T depends on A
logit = -1.5 + 1.6*T + 1.6*B + 1.6*S
C = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)            # collider child
df = pd.DataFrame({"A": A, "B": B, "S": S, "T": T, "C": C, "D": D})
true_mb = {"A", "C", "B", "S"}

def report(x, y, z=None):
    z = z or []
    _, _, p = g_test(df[x].to_numpy(), df[y].to_numpy(), [df[v].to_numpy() for v in z])
    z_str = f" | {z}" if z else ""
    print(f"test({x};{y}{z_str}):  p={p:.3e}  -> {'INDEPENDENT' if p > ALPHA else 'dependent'}")

print("True MB(T):", sorted(true_mb))
df.head()

True MB(T): ['A', 'B', 'C', 'S']


,A,B,S,T,C,D
0,0,1,0,0,0,0
1,1,0,1,1,0,1
2,1,1,0,1,1,1
3,1,1,0,1,1,0
4,0,0,0,1,1,0


## 2. Parent and child are obviously dependent on T

In [5]:
report("A", "T")   # parent
report("C", "T")   # child
report("D", "T")   # noise

test(A;T):  p=0.000e+00  -> dependent
test(C;T):  p=4.950e-189  -> dependent
test(D;T):  p=8.231e-01  -> INDEPENDENT


> **Note:** if `D` occasionally shows p just under 0.05, that's a *false
> positive* — a single test at α=0.05 wrongly rejects ~5% of the time. Re-run
> with another seed and it usually flips. This is exactly why discovery
> algorithms have a **shrink phase** rather than trusting one test.


## 3. The spouse is invisible — until you condition on the child

A spouse has **no direct edge** to `T`, so marginally they're independent. But
they share a child `C`; conditioning on that collider **opens** the path, making
them dependent. That's why the spouse must be in the blanket.


In [6]:
report("S", "T")            # marginally INDEPENDENT
report("S", "T", ["C"])     # dependent once we condition on shared child C
print()
report("B", "T")            # same for the other spouse
report("B", "T", ["C"])

test(S;T):  p=9.723e-01  -> INDEPENDENT
test(S;T | ['C']):  p=1.703e-19  -> dependent

test(B;T):  p=8.573e-02  -> INDEPENDENT
test(B;T | ['C']):  p=6.651e-14  -> dependent


## 4. The blanket does its job

Conditioned on the full blanket `{A, C, B, S}`, the target `T` is independent of
the only remaining variable `D` — the defining property of a Markov Blanket.


In [7]:
report("D", "T", ["A", "C", "B", "S"])

test(D;T | ['A', 'C', 'B', 'S']):  p=9.575e-01  -> INDEPENDENT


## 5. Why "top-k by mutual information" fails

A naive filter ranks features by marginal `I(feature; T)`. It scores the spouses
`B`, `S` near zero (they're marginally independent of T) and would drop them —
yet they carry real information *in combination with C*. MB selection captures
exactly this interaction.


In [8]:
print("Marginal I(feature; T) — what a naive filter ranks by:")
for col in ["A", "C", "B", "S", "D"]:
    flag = "  <- spouse, looks irrelevant marginally!" if col in ("B", "S") else ""
    print(f"  I({col};T) = {mutual_information(df[col].to_numpy(), df['T'].to_numpy()):.4f}{flag}")

Marginal I(feature; T) — what a naive filter ranks by:
  I(A;T) = 0.1892
  I(C;T) = 0.0430
  I(B;T) = 0.0001  <- spouse, looks irrelevant marginally!
  I(S;T) = 0.0000  <- spouse, looks irrelevant marginally!
  I(D;T) = 0.0000


### Takeaways

- `MB(T) = parents ∪ children ∪ spouses`.
- A spouse is **marginally independent** of T but **dependent given the shared
  child** — colliders open paths when conditioned on.
- Conditioning on the full blanket makes T independent of all else.
- Marginal-MI filters miss spouses; MB selection captures the interaction —
  the core advantage of the whole approach.

**Exercises:** `exercises/03_graphical_models.md`, `exercises/04_markov_blanket_theory.md`.
